# Цифровая обработка изображений
## Лабораторная работа 3. Коррекция точки белого
**Студент:** Малинин Даниил Дмитриевич  
**Группа:** 6131

## Раздел 0. Загрузка исходных изображений

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def load_rgb(path):
    bgr = cv2.imread(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

img1 = load_rgb('lab_03_1.jpg')  # тёплый cast
img2 = load_rgb('lab_03_2.jpg')  # холодный/синий cast

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img1); axes[0].set_title('lab_03_1.jpg (тёплый тон)'); axes[0].axis('off')
axes[1].imshow(img2); axes[1].set_title('lab_03_2.jpg (синий cast)'); axes[1].axis('off')
plt.suptitle('Исходные изображения с плохим балансом белого')
plt.tight_layout(); plt.show()

print(f'Разрешение: {img1.shape[1]}x{img1.shape[0]}')
print(f'Средние R/G/B (img1): {img1[:,:,0].mean():.1f} / {img1[:,:,1].mean():.1f} / {img1[:,:,2].mean():.1f}')
print(f'Средние R/G/B (img2): {img2[:,:,0].mean():.1f} / {img2[:,:,1].mean():.1f} / {img2[:,:,2].mean():.1f}')

## Задание 1. Ручная коррекция по серой карте

Написать алгоритм, который корректирует баланс белого, используя известные значения «серого» пикселя (или области) в изображении. Учитывать, что серый пиксель может иметь яркость, отличную от белого.

In [ ]:
def white_balance_gray_patch(img, patch_coords, target_gray=None):
    """
    img: RGB uint8
    patch_coords: (y1, y2, x1, x2) — координаты «серой» области
    target_gray: целевая яркость серого (если None — среднее по каналу G)
    """
    y1, y2, x1, x2 = patch_coords
    patch = img[y1:y2, x1:x2].astype(np.float32)
    
    avg_r = patch[:,:,0].mean()
    avg_g = patch[:,:,1].mean()
    avg_b = patch[:,:,2].mean()
    
    if target_gray is None:
        target_gray = avg_g  # нормируем по зелёному каналу
    
    scale_r = target_gray / avg_r if avg_r > 0 else 1.0
    scale_g = target_gray / avg_g if avg_g > 0 else 1.0
    scale_b = target_gray / avg_b if avg_b > 0 else 1.0
    
    result = img.astype(np.float32).copy()
    result[:,:,0] *= scale_r
    result[:,:,1] *= scale_g
    result[:,:,2] *= scale_b
    
    return np.clip(result, 0, 255).astype(np.uint8), (scale_r, scale_g, scale_b)


# Координаты «серой» области — нижняя треть центральной части (область зубов/светлой кожи)
patch1 = (img1.shape[0]*2//3, img1.shape[0]*3//4, img1.shape[1]//3, img1.shape[1]*2//3)
patch2 = (img2.shape[0]*2//3, img2.shape[0]*3//4, img2.shape[1]//3, img2.shape[1]*2//3)

result1_gp, scales1_gp = white_balance_gray_patch(img1, patch1)
result2_gp, scales2_gp = white_balance_gray_patch(img2, patch2)

print('Коэффициенты масштабирования (scale_R, scale_G, scale_B):')
print(f'  img1 (тёплый): R={scales1_gp[0]:.3f}, G={scales1_gp[1]:.3f}, B={scales1_gp[2]:.3f}')
print(f'  img2 (синий):  R={scales2_gp[0]:.3f}, G={scales2_gp[1]:.3f}, B={scales2_gp[2]:.3f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].imshow(img1); axes[0,0].set_title('img1 — исходное (тёплый)'); axes[0,0].axis('off')
axes[0,1].imshow(result1_gp); axes[0,1].set_title('img1 — после коррекции (серая карта)'); axes[0,1].axis('off')
axes[1,0].imshow(img2); axes[1,0].set_title('img2 — исходное (синий)'); axes[1,0].axis('off')
axes[1,1].imshow(result2_gp); axes[1,1].set_title('img2 — после коррекции (серая карта)'); axes[1,1].axis('off')
plt.suptitle('Задание 1. Коррекция по серой карте')
plt.tight_layout(); plt.show()

**Вывод:** Метод коррекции по серой карте опирается на предположение, что выбранная область должна иметь одинаковые значения по всем трём каналам. Для тёплого изображения (`img1`) красный канал в серой области преобладает над синим, поэтому `scale_r < 1` (красный нужно уменьшить) и `scale_b > 1` (синий нужно усилить). Для холодного изображения (`img2`) ситуация обратная: синий канал избыточен, поэтому `scale_b < 1`, а `scale_r > 1`. Точность метода напрямую зависит от правильного выбора серой области — ошибка при выборе участка приводит к неверной коррекции всего снимка. Метод хорошо работает при наличии нейтральной области с известной яркостью.

## Задание 2. Автоматический баланс белого (ярчайшая точка/область)

Написать алгоритм, который находит самую яркую точку/область заданного размера и использует её для коррекции баланса белого.

In [ ]:
def white_balance_brightest(img, patch_size=20, target_val=255):
    """
    Находит область patch_size x patch_size с максимальной суммарной яркостью.
    Использует её как «белый» эталон.
    """
    img_f = img.astype(np.float32)
    # Суммарная яркость по каналам
    brightness = img_f.sum(axis=2)
    # Скользящее среднее для поиска наиярчайшей области
    from scipy.ndimage import uniform_filter
    brightness_smooth = uniform_filter(brightness, size=patch_size)
    
    # Координаты наиярчайшей области
    idx = np.unravel_index(brightness_smooth.argmax(), brightness_smooth.shape)
    cy, cx = idx
    h2 = patch_size // 2
    y1, y2 = max(0, cy-h2), min(img.shape[0], cy+h2)
    x1, x2 = max(0, cx-h2), min(img.shape[1], cx+h2)
    
    patch = img_f[y1:y2, x1:x2]
    avg_r = patch[:,:,0].mean()
    avg_g = patch[:,:,1].mean()
    avg_b = patch[:,:,2].mean()
    
    scale_r = target_val / avg_r if avg_r > 0 else 1.0
    scale_g = target_val / avg_g if avg_g > 0 else 1.0
    scale_b = target_val / avg_b if avg_b > 0 else 1.0
    
    result = img_f.copy()
    result[:,:,0] *= scale_r
    result[:,:,1] *= scale_g
    result[:,:,2] *= scale_b
    
    result_img = np.clip(result, 0, 255).astype(np.uint8)
    return result_img, (y1, y2, x1, x2), (scale_r, scale_g, scale_b)


result1_bp, coords1, scales1_bp = white_balance_brightest(img1)
result2_bp, coords2, scales2_bp = white_balance_brightest(img2)

print('Коэффициенты масштабирования (scale_R, scale_G, scale_B):')
print(f'  img1 (тёплый): R={scales1_bp[0]:.3f}, G={scales1_bp[1]:.3f}, B={scales1_bp[2]:.3f}')
print(f'  img2 (синий):  R={scales2_bp[0]:.3f}, G={scales2_bp[1]:.3f}, B={scales2_bp[2]:.3f}')
print(f'  Найденная область img1: y=[{coords1[0]}:{coords1[1]}], x=[{coords1[2]}:{coords1[3]}]')
print(f'  Найденная область img2: y=[{coords2[0]}:{coords2[1]}], x=[{coords2[2]}:{coords2[3]}]')

# Копии с прямоугольниками для визуализации
img1_rect = img1.copy()
img2_rect = img2.copy()
cv2.rectangle(img1_rect, (coords1[2], coords1[0]), (coords1[3], coords1[1]), (0, 255, 0), 3)
cv2.rectangle(img2_rect, (coords2[2], coords2[0]), (coords2[3], coords2[1]), (0, 255, 0), 3)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0,0].imshow(img1); axes[0,0].set_title('img1 — исходное (тёплый)'); axes[0,0].axis('off')
axes[0,1].imshow(img1_rect); axes[0,1].set_title('img1 — найденная область (зелёный прямоугольник)'); axes[0,1].axis('off')
axes[0,2].imshow(result1_bp); axes[0,2].set_title('img1 — после коррекции (ярчайшая точка)'); axes[0,2].axis('off')
axes[1,0].imshow(img2); axes[1,0].set_title('img2 — исходное (синий)'); axes[1,0].axis('off')
axes[1,1].imshow(img2_rect); axes[1,1].set_title('img2 — найденная область (зелёный прямоугольник)'); axes[1,1].axis('off')
axes[1,2].imshow(result2_bp); axes[1,2].set_title('img2 — после коррекции (ярчайшая точка)'); axes[1,2].axis('off')
plt.suptitle('Задание 2. Коррекция по ярчайшей точке/области')
plt.tight_layout(); plt.show()

**Вывод:** Метод ярчайшей точки основан на предположении, что самый яркий фрагмент изображения соответствует белому цвету. Алгоритм автоматически ищет область с максимальной суммарной яркостью и нормирует каналы так, чтобы эта область стала белой (255, 255, 255). Метод полностью автоматический и не требует ручного указания эталонной области, однако чувствителен к наличию ярких цветных объектов — если самая яркая область не является белой, коррекция будет ошибочной. На портрете ярчайшими, как правило, оказываются блики на коже или зубы, что даёт приемлемый результат. Зелёным прямоугольником выделена область, использованная как эталон белого.

## Задание 3. Gray World Algorithm

Реализовать алгоритм Gray World, который предполагает, что средний цвет изображения — серый.

In [ ]:
def white_balance_gray_world(img):
    """
    Предполагаем, что среднее по всем каналам должно быть равным.
    Нормируем по среднему значению зелёного канала.
    """
    img_f = img.astype(np.float32)
    avg_r = img_f[:,:,0].mean()
    avg_g = img_f[:,:,1].mean()
    avg_b = img_f[:,:,2].mean()
    
    # Нормируем так, чтобы все три средних стали равны avg_g
    scale_r = avg_g / avg_r if avg_r > 0 else 1.0
    scale_b = avg_g / avg_b if avg_b > 0 else 1.0
    
    result = img_f.copy()
    result[:,:,0] *= scale_r
    result[:,:,2] *= scale_b
    
    return np.clip(result, 0, 255).astype(np.uint8), (scale_r, 1.0, scale_b)


result1_gw, scales1_gw = white_balance_gray_world(img1)
result2_gw, scales2_gw = white_balance_gray_world(img2)

print('Средние значения каналов до коррекции:')
print(f'  img1: R={img1[:,:,0].mean():.1f}, G={img1[:,:,1].mean():.1f}, B={img1[:,:,2].mean():.1f}')
print(f'  img2: R={img2[:,:,0].mean():.1f}, G={img2[:,:,1].mean():.1f}, B={img2[:,:,2].mean():.1f}')
print('Коэффициенты масштабирования (scale_R, scale_G, scale_B):')
print(f'  img1 (тёплый): R={scales1_gw[0]:.3f}, G={scales1_gw[1]:.3f}, B={scales1_gw[2]:.3f}')
print(f'  img2 (синий):  R={scales2_gw[0]:.3f}, G={scales2_gw[1]:.3f}, B={scales2_gw[2]:.3f}')
print('Средние значения каналов после коррекции:')
print(f'  img1: R={result1_gw[:,:,0].mean():.1f}, G={result1_gw[:,:,1].mean():.1f}, B={result1_gw[:,:,2].mean():.1f}')
print(f'  img2: R={result2_gw[:,:,0].mean():.1f}, G={result2_gw[:,:,1].mean():.1f}, B={result2_gw[:,:,2].mean():.1f}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].imshow(img1); axes[0,0].set_title('img1 — исходное (тёплый)'); axes[0,0].axis('off')
axes[0,1].imshow(result1_gw); axes[0,1].set_title('img1 — после коррекции (Gray World)'); axes[0,1].axis('off')
axes[1,0].imshow(img2); axes[1,0].set_title('img2 — исходное (синий)'); axes[1,0].axis('off')
axes[1,1].imshow(result2_gw); axes[1,1].set_title('img2 — после коррекции (Gray World)'); axes[1,1].axis('off')
plt.suptitle('Задание 3. Алгоритм Gray World')
plt.tight_layout(); plt.show()

**Вывод:** Алгоритм Gray World основан на статистическом предположении: если сцена содержит достаточно разнообразные по цвету объекты, то среднее значение по каждому каналу должно быть одинаковым. Метод полностью автоматический и вычислительно дешёвый — достаточно одного прохода по изображению для вычисления средних. По сравнению с методом серой карты Gray World менее точен при сценах с явным цветовым преобладанием (например, портреты на однотонном фоне), но не требует знания расположения нейтральной области. По сравнению с методом ярчайшей точки он менее чувствителен к выбросам (одиночным ярким пикселям), поскольку усредняет всё изображение целиком.

## Задание 4*. Коррекция через цветовую температуру (в Кельвинах)

Написать функцию, которая применяет коррекцию, зная цветовую температуру освещения. Использовать приближённые формулы для перевода температуры в RGB.

In [ ]:
def kelvin_to_rgb(temp_k):
    """
    Аппроксимация Tanner Helland для перевода цветовой температуры в RGB [0..255].
    Диапазон: 1000K–40000K.
    """
    temp = temp_k / 100.0
    
    if temp <= 66:
        r = 255
    else:
        r = 329.698727446 * ((temp - 60) ** -0.1332047592)
        r = np.clip(r, 0, 255)
    
    if temp <= 66:
        g = 99.4708025861 * np.log(temp) - 161.1195681661
    else:
        g = 288.1221695283 * ((temp - 60) ** -0.0755148492)
    g = np.clip(g, 0, 255)
    
    if temp >= 66:
        b = 255
    elif temp <= 19:
        b = 0
    else:
        b = 138.5177312231 * np.log(temp - 10) - 305.0447927307
        b = np.clip(b, 0, 255)
    
    return np.array([r, g, b], dtype=np.float32)


def white_balance_kelvin(img, source_temp_k, target_temp_k=6500):
    """
    Корректирует баланс белого, зная температуру источника освещения.
    source_temp_k: температура на снимке
    target_temp_k: целевая температура (по умолчанию 6500K — дневной свет)
    """
    source_rgb = kelvin_to_rgb(source_temp_k)
    target_rgb  = kelvin_to_rgb(target_temp_k)
    
    scale = target_rgb / (source_rgb + 1e-6)
    
    img_f = img.astype(np.float32)
    result = img_f * scale[np.newaxis, np.newaxis, :]
    return np.clip(result, 0, 255).astype(np.uint8), scale


# Применяем коррекцию
result1_kv, scale1_kv = white_balance_kelvin(img1, source_temp_k=3000, target_temp_k=6500)
result2_kv, scale2_kv = white_balance_kelvin(img2, source_temp_k=7500, target_temp_k=6500)

print('RGB-значения цветовых температур:')
print(f'  3000K (лампа накаливания): R={kelvin_to_rgb(3000)[0]:.1f}, G={kelvin_to_rgb(3000)[1]:.1f}, B={kelvin_to_rgb(3000)[2]:.1f}')
print(f'  6500K (дневной свет):      R={kelvin_to_rgb(6500)[0]:.1f}, G={kelvin_to_rgb(6500)[1]:.1f}, B={kelvin_to_rgb(6500)[2]:.1f}')
print(f'  7500K (пасмурно):          R={kelvin_to_rgb(7500)[0]:.1f}, G={kelvin_to_rgb(7500)[1]:.1f}, B={kelvin_to_rgb(7500)[2]:.1f}')
print()
print('Коэффициенты масштабирования (scale_R, scale_G, scale_B):')
print(f'  img1 (3000K → 6500K): R={scale1_kv[0]:.3f}, G={scale1_kv[1]:.3f}, B={scale1_kv[2]:.3f}')
print(f'  img2 (7500K → 6500K): R={scale2_kv[0]:.3f}, G={scale2_kv[1]:.3f}, B={scale2_kv[2]:.3f}')

# График kelvin_to_rgb для диапазона 1000K–10000K
temps = np.arange(1000, 10001, 100)
rgb_vals = np.array([kelvin_to_rgb(t) for t in temps])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(temps, rgb_vals[:,0], 'r-', label='Красный (R)', linewidth=2)
axes[0].plot(temps, rgb_vals[:,1], 'g-', label='Зелёный (G)', linewidth=2)
axes[0].plot(temps, rgb_vals[:,2], 'b-', label='Синий (B)', linewidth=2)
axes[0].axvline(x=3000, color='orange', linestyle='--', alpha=0.7, label='3000K (лампа)')
axes[0].axvline(x=6500, color='gray', linestyle='--', alpha=0.7, label='6500K (дневной)')
axes[0].axvline(x=7500, color='cyan', linestyle='--', alpha=0.7, label='7500K (пасмурно)')
axes[0].set_xlabel('Цветовая температура (K)')
axes[0].set_ylabel('Значение канала (0–255)')
axes[0].set_title('RGB-компоненты цветовой температуры (формула Tanner Helland)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Цветовая полоса температур
color_strip = np.array([[kelvin_to_rgb(t) / 255.0 for t in temps]])
axes[1].imshow(color_strip, aspect='auto', extent=[1000, 10000, 0, 1])
axes[1].set_xlabel('Цветовая температура (K)')
axes[1].set_yticks([])
axes[1].set_title('Визуализация цветов по температуре')

plt.tight_layout(); plt.show()

# Результаты коррекции
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].imshow(img1); axes[0,0].set_title('img1 — исходное (3000K, тёплый)'); axes[0,0].axis('off')
axes[0,1].imshow(result1_kv); axes[0,1].set_title('img1 — после коррекции до 6500K'); axes[0,1].axis('off')
axes[1,0].imshow(img2); axes[1,0].set_title('img2 — исходное (7500K, синий)'); axes[1,0].axis('off')
axes[1,1].imshow(result2_kv); axes[1,1].set_title('img2 — после коррекции до 6500K'); axes[1,1].axis('off')
plt.suptitle('Задание 4*. Коррекция через цветовую температуру (Кельвины)')
plt.tight_layout(); plt.show()

**Вывод:** Метод коррекции через цветовую температуру использует физическую модель излучения абсолютно чёрного тела, аппроксимированную формулой Tanner Helland. Зная температуру источника освещения (3000K для лампы накаливания, 7500K для пасмурного дня) и желаемую целевую температуру (6500K — стандартный дневной свет), можно вычислить точные коэффициенты масштабирования каналов. Для тёплого снимка (`img1`, 3000K) красный канал необходимо уменьшить, а синий усилить (`scale_r < 1`, `scale_b > 1`). Для холодного снимка (`img2`, 7500K) красный канал усиливается, а синий ослабляется. Метод физически обоснован, однако требует точного знания температуры источника, что не всегда доступно.

## Итоговое сравнение всех методов

In [ ]:
methods = [
    'Исходное',
    'Серая карта',
    'Ярчайшая\nточка',
    'Gray World',
    'Кельвин'
]

results_img1 = [img1, result1_gp, result1_bp, result1_gw, result1_kv]
results_img2 = [img2, result2_gp, result2_bp, result2_gw, result2_kv]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

row_labels = ['img1 (тёплый, 3000K)', 'img2 (синий, 7500K)']
all_results = [results_img1, results_img2]

for row_idx, (row_results, row_label) in enumerate(zip(all_results, row_labels)):
    for col_idx, (result, method_name) in enumerate(zip(row_results, methods)):
        ax = axes[row_idx, col_idx]
        ax.imshow(result)
        ax.axis('off')
        if row_idx == 0:
            ax.set_title(method_name, fontsize=11, fontweight='bold')
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=10, fontweight='bold', labelpad=10)
            ax.yaxis.set_label_position('left')
            ax.tick_params(left=False, labelleft=False)

plt.suptitle('Итоговое сравнение всех методов коррекции баланса белого', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Сводная таблица коэффициентов
print('=' * 70)
print('Сводная таблица коэффициентов масштабирования каналов R/G/B')
print('=' * 70)
print(f'{"Метод":<20} {"img1 R":>8} {"img1 G":>8} {"img1 B":>8}   {"img2 R":>8} {"img2 G":>8} {"img2 B":>8}')
print('-' * 70)
all_scales = [
    ('Серая карта',    scales1_gp,  scales2_gp),
    ('Ярчайшая точка', scales1_bp,  scales2_bp),
    ('Gray World',     scales1_gw,  scales2_gw),
    ('Кельвин',        tuple(scale1_kv), tuple(scale2_kv)),
]
for name, s1, s2 in all_scales:
    print(f'{name:<20} {s1[0]:>8.3f} {s1[1]:>8.3f} {s1[2]:>8.3f}   {s2[0]:>8.3f} {s2[1]:>8.3f} {s2[2]:>8.3f}')
print('=' * 70)

**Вывод:** Все четыре рассмотренных метода успешно устраняют цветовой сдвиг на обоих изображениях, однако каждый из них имеет свои особенности. Метод серой карты даёт наиболее точный результат при правильно выбранной эталонной области, но требует ручного вмешательства. Метод ярчайшей точки полностью автоматизирован, но может ошибаться, если самый яркий фрагмент не является белым. Алгоритм Gray World прост и универсален, но менее эффективен для сцен с явным цветовым доминированием. Метод Кельвина физически наиболее обоснован и даёт предсказуемые результаты, однако требует знания цветовой температуры источника освещения. На практике для портретной съёмки лучшие результаты обычно показывают метод серой карты (при наличии нейтрального элемента) и метод Кельвина (при известных условиях освещения).